# Proyecto AEMET - Despliegue en AWS EC2, NLP y FastAPI
---
Este notebook resume y explica el trabajo de integración y despliegue del proyecto. 
La meta de esta fase ha sido **coger todas las piezas individuales** (el ETL, el frontend interactivo, los modelos de NLP y XGBoost) y **conectarlas en una arquitectura robusta** desplegada en una máquina virtual de la nube (AWS EC2).

---
## 📋 Objetivos de este Notebook
1. **Organización del Código:** Explicar la estructura de archivos de producción en la nube.
2. **Arquitectura Final:** Mostrar el flujo de datos escalable entre el Frontend, Backend (FastAPI) y la Base de Datos.
3. **Demostración NLP (Text-to-SQL):** Validar la integración del LLM (Google Gemini) respondiendo preguntas en lenguaje natural.
4. **Consulta de Históricos y Predicciones:** Mostrar cómo la API sirve datos históricos para gráficos y predicciones de Machine Learning (XGBoost) en tiempo real.
5. **Conclusión:** Resumir las ventajas de la arquitectura implementada.

---
### 📋 Imports necesarios para el Despliegue y Pruebas 📋
---
### 📁 Sistema Operativo y Entorno
* **`os`**: Gestionar variables del sistema y entorno.
* **`dotenv.load_dotenv`**: Cargar configuraciones de forma segura desde el archivo `.env`.

### 📊 Procesamiento y Análisis de Datos
* **`pandas` (como `pd`)**: Manejar los datos estructurados devueltos por la API en forma de DataFrames.

### 🌐 Conexión Web y APIs
* **`requests`**: Realizar peticiones HTTP a nuestra API REST desplegada en EC2.

### 🖥️ Visualización de Datos
* **`matplotlib.pyplot` / `seaborn`**: Generar gráficos para validar los históricos climáticos.
* **`IPython.display` (`display`)**: Mostrar DataFrames con un formato amigable.

In [ ]:
# Imports necesarios para el Proyecto AEMET (Despliegue)

#================================================================================#
#               SISTEMA OPERATIVO Y ENTORNO SEGURO                               # 
#================================================================================#
import os
from dotenv import load_dotenv

#================================================================================#
#                 PROCESAMIENTO DE DATOS Y CONEXIÓN WEB                          # 
#================================================================================#
import pandas as pd
import requests

#================================================================================#
#               INTERFAZ Y VISUALIZACIÓN                                         # 
#================================================================================#
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Ocultar warnings visuales
import warnings
warnings.filterwarnings('ignore')

# Configuración visual base para gráficos
sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams['figure.figsize'] = (10, 5)

print("Librerías de análisis y visualización cargadas correctamente.")

## Paso 1: Configuración del Entorno de Conexión

Para simular de forma realista el comportamiento del Frontend, necesitamos la URL base donde nuestra API FastAPI está escuchando en AWS EC2.
Utilizaremos el archivo `.env` para cargar esta IP o, en su defecto, usaremos la IP pública actual por defecto.

In [ ]:
# Carga de variables de entorno (simulando lo que hace el frontend)
load_dotenv()

# Obtenemos la URL de FastAPI desde el entorno, o usamos la IP de EC2 por defecto
FASTAPI_URL = os.getenv("FASTAPI_URL", "http://18.198.208.67:8000")

print(f"🔗 Conectando a la API en: {FASTAPI_URL}")

## Paso 2: Organización del Código y Arquitectura en la Nube

Para evaluar qué código está corriendo en nuestro servidor AWS EC2, todos los archivos de producción están en la carpeta `Despliegue_EC2/`.
Destacan:
- `api_unificada.py`: El backend en **FastAPI** (conexión a PostgreSQL, modelos XGBoost, LLM Gemini).
- `app_unificada.py`: La interfaz en **Streamlit**.
- `modelos_max/` y `modelos_min/`: Modelos `.pkl` preentrenados.

### Arquitectura Final Implementada

El flujo separa responsabilidades, haciendo la web más ligera y segura:

```mermaid
graph LR
    A[Usuario Final] -->|Interactúa (Puerto 8501)| B(Streamlit - app_unificada)
    B -->|HTTP GET/POST (Puerto 8000)| C{FastAPI Backend - api_unificada}
    C -->|Consultas SQL (Puerto 5432)| D[(AWS RDS PostgreSQL)]
    C -->|API REST| E[Google Gemini LLM]
    C -->|Inferencia| F[Modelos ML Locales]
```

## Paso 3: Demostración del Motor NLP (Text-to-SQL)

El modelo de Lenguaje Natural (Google Gemini) actúa como un analista virtual. 
A través de LangChain y Pydantic, el backend recibe una pregunta, genera el código SQL adaptado a nuestro esquema, ejecuta la consulta y devuelve los registros.

In [ ]:
pregunta_usuario = "¿Cuáles fueron las 5 temperaturas máximas más altas registradas en Madrid durante el verano de 2023?"
print(f"👤 Pregunta del usuario: '{pregunta_usuario}'\n")

try:
    print("⏳ Enviando pregunta a la API para procesar con Gemini...")
    # Simulamos la petición POST que hace Streamlit por debajo
    respuesta_nlp = requests.post(f"{FASTAPI_URL}/ask", json={"pregunta": pregunta_usuario}, timeout=20)
    
    if respuesta_nlp.status_code == 200:
        datos_respuesta = respuesta_nlp.json()
        print("✅ Traducción a SQL y consulta a BD completadas.\n")
        print(f"📝 SQL Generado internamente:\n{datos_respuesta.get('sql_generado')}\n")
        
        # Convertimos la respuesta a un DataFrame para visualizarlo
        df_nlp = pd.DataFrame(datos_respuesta.get('datos', []))
        print("📊 Resultados obtenidos listos para mostrar:")
        display(df_nlp.head())
    else:
        print(f"❌ Error de la API: {respuesta_nlp.status_code} - {respuesta_nlp.text}")

except requests.exceptions.Timeout:
    print("❌ Error: La API tardó demasiado en responder (Timeout).")
except requests.exceptions.RequestException as e:
    print(f"❌ Error de conexión a EC2: {e}")

## Paso 4: Consumo de Históricos e Inferencia ML (XGBoost)

La API también sirve los modelos predictivos y el histórico de datos. 
A continuación, validaremos la extracción de datos de una estación en un mes concreto y pediremos al modelo que prediga la temperatura máxima basándose en la última lectura.

In [ ]:
indicativo_estacion = "3195" # Estación Madrid, Retiro
fecha_ini_prueba = "2024-01-01"
fecha_fin_prueba = "2024-01-31"

try:
    # 1. Obtenemos el histórico
    print(f"📅 Obteniendo histórico para la estación {indicativo_estacion} ({fecha_ini_prueba} al {fecha_fin_prueba})...")
    res_hist = requests.get(f"{FASTAPI_URL}/historico/obtener_historico", 
                            params={"id": indicativo_estacion, "fecha_inicio": fecha_ini_prueba, "fecha_fin": fecha_fin_prueba},
                            timeout=15)
    
    if res_hist.status_code == 200:
        # Cargamos y preparamos los datos
        df_hist = pd.DataFrame(res_hist.json().get("registros", []))
        if not df_hist.empty:
            df_hist['fecha'] = pd.to_datetime(df_hist['fecha'])
            
            # Graficamos la evolución térmica
            plt.figure(figsize=(12, 5))
            sns.lineplot(data=df_hist, x='fecha', y='tmax', marker="o", label="Temp. Máxima", color="#ff6b6b")
            sns.lineplot(data=df_hist, x='fecha', y='tmed', marker="s", label="Temp. Media", color="#4ecdc4")
            sns.lineplot(data=df_hist, x='fecha', y='tmin', marker="^", label="Temp. Mínima", color="#45b7d1")
            
            plt.title(f"Evolución Térmica (API) - Estación {indicativo_estacion}", fontsize=14, pad=15)
            plt.xlabel("Fecha")
            plt.ylabel("Temperatura (ºC)")
            plt.legend()
            plt.show()
        else:
            print("⚠️ La API no devolvió registros para este rango de fechas.")
        
        # 2. Petición al modelo de ML
        print("\n🤖 Pidiendo inferencia en tiempo real al modelo XGBoost...")
        res_ult = requests.get(f"{FASTAPI_URL}/historico/ultima_tmed", params={"id": indicativo_estacion}, timeout=10)
        
        if res_ult.status_code == 200:
            ultima_tmed = res_ult.json().get('tmed')
            if ultima_tmed is not None:
                # Enviamos la característica (feature) al modelo para predecir
                res_pred = requests.post(f"{FASTAPI_URL}/modelos_max/{indicativo_estacion}/predict", 
                                         json={"features": [ultima_tmed]}, timeout=10)
                
                if res_pred.status_code == 200:
                    prediccion = res_pred.json().get('prediccion', [[None]])[0][0]
                    print(f"✅ Con una T. Media actual de {ultima_tmed}ºC, el modelo predice una T. Máxima de {prediccion:.2f}ºC")
                else:
                    print(f"❌ Error al predecir: {res_pred.status_code}")
            else:
                print("⚠️ No se pudo obtener la última temperatura media.")
        else:
            print(f"❌ Error al obtener la última medición: {res_ult.status_code}")
    else:
         print(f"❌ Error al obtener histórico: {res_hist.status_code} - {res_hist.text}")
            
except requests.exceptions.RequestException as e:
    print(f"❌ Error general de red: {e}")

## Paso 5: Conclusión de la Integración

El principal reto de esta fase ha sido **desacoplar el frontend del acceso a datos**. Inicialmente, los prototipos accedían directamente a la base de datos o ejecutaban los modelos en memoria.

Al implementar **FastAPI** como *Middleware* en **EC2**:
1. **Seguridad Centralizada:** Las credenciales de la BD (RDS) y de Gemini solo existen en el backend.
2. **Rendimiento:** FastAPI carga los modelos de Machine Learning pesados en memoria una sola vez al arrancar, evitando retrasos al usuario.
3. **Frontend Ligero:** Permite que **Streamlit** se dedique exclusivamente a gestionar la UI e interacciones, renderizando los resultados que recibe por HTTP.

*(El código completo en producción se encuentra en los scripts `.py` de la carpeta `Despliegue_EC2/`)*.